In [ ]:
# 1. IMPORT LIBRARIES, CONFIGURE PATHS & ENVIRONMENT SETUP
# ------------------------------------------------------------

%pip install python-dotenv google-genai

import json
import time
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
from google import genai
from IPython.display import Markdown, display
import os

# PROJECT PATH SETUP
PROJECT_ROOT = Path(r"E:\STREAMINTEL360_Complete")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

ARTIFACT_ROOT = ARTIFACTS_DIR / "notebook_06_llm"
METRICS_DIR = ARTIFACT_ROOT / "metrics"
REPORTS_DIR = ARTIFACT_ROOT / "reports"
METADATA_DIR = ARTIFACT_ROOT / "metadata"

for dir_path in [METRICS_DIR, REPORTS_DIR, METADATA_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Loading .env and Gemini API key
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        f"Could not load GEMINI_API_KEY from {ENV_PATH}. "
        "Please check that the .env file exists and contains GEMINI_API_KEY.")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = "gemini-3.1-flash-lite"

PROJECT_NAME = "STREAMINTEL 360"
MODULE_NAME = "LLM Intelligence"

print("STREAMINTEL 360 — LLM ENVIRONMENT INITIALIZATION")
print(f"Project       : {PROJECT_NAME}")
print(f"Module        : {MODULE_NAME}")
print(f"LLM Provider  : Google Gemini")
print(f"Model         : {MODEL_NAME}")
print(f"Artifacts Dir : {ARTIFACTS_DIR.resolve()}")
print(f"Artifact Root : {ARTIFACT_ROOT.resolve()}")
print("API Key       : Successfully loaded")


  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached google_genai-2.18.1-py3-none-any.whl.metadata (56 kB)
  Using cached google_auth-2.56.3-py3-none-any.whl.metadata (6.0 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached pyasn1-0.6.4-py3-none-any.whl.metadata (8.4 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Using cached google_genai-2.18.1-py3-none-any.whl (1.1 MB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached google_auth-2.56.3-py3-none-any.whl (259 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   --------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


STREAMINTEL 360 — LLM ENVIRONMENT INITIALIZATION
Project       : STREAMINTEL 360
Module        : LLM Intelligence
LLM Provider  : Google Gemini
Model         : gemini-3.1-flash-lite
Artifacts Dir : E:\STREAMINTEL360_Complete\artifacts
Artifact Root : E:\STREAMINTEL360_Complete\artifacts\notebook_06_llm
API Key       : Successfully loaded


In [ ]:
# 2. EVIDENCE-GROUNDED ANALYTICS AGGREGATION
# ------------------------------------------------------------

def _load_json(path: Path, required: bool = True):
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Required evidence file not found: {path}")
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_and_aggregate_metrics() -> dict:

    # Notebook 01: Subscriber Churn 
    churn_dir = ARTIFACTS_DIR / "notebook_01_churn"
    churn_metadata = _load_json(churn_dir / "metadata" / "metadata.json")
    churn_comparison = _load_json(churn_dir / "metrics" / "model_comparison.json")

    # Notebook 02: Demand Forecasting
    forecast_dir = ARTIFACTS_DIR / "notebook_02_forecasting"
    forecast_comparison = _load_json(forecast_dir / "metrics" / "model_comparison.json")
    # model_comparison.json is a list of per-model records (Model, MAE, RMSE, ...)
    forecast_by_model = {row["Model"]: row for row in forecast_comparison} if forecast_comparison else {}
    best_forecast_model = min(forecast_by_model, key=lambda m: forecast_by_model[m].get("RMSE", float("inf"))) if forecast_by_model else None

    # Notebook 03: Recommendation Engine
    rec_dir = ARTIFACTS_DIR / "notebook_03_recommendations"
    hybrid_metrics = _load_json(rec_dir / "metrics" / "hybrid_evaluation_metrics.json")
    optimal_weights = _load_json(rec_dir / "metrics" / "optimal_hybrid_weights.json")

    # Notebook 04: Computer Vision
    cv_dir = ARTIFACTS_DIR / "notebook_04_computer_vision"
    cv_inference_config = _load_json(cv_dir / "models" / "efficientnetb0_inference_config.json")

    # Notebook 05: NLP & Sentiment
    nlp_dir = ARTIFACTS_DIR / "notebook_05_nlp_sentiment"
    nlp_metadata = _load_json(nlp_dir / "models" / "streamintel_sentiment_metadata.json")

    analytics_payload = {
        "architecture_note": {
            "pipeline_type": "LLM Reasoning & Executive Synthesis (Inference Only)",
            "description": (
                "Notebook 06 consumes previously generated ML/DL artifacts "
                "and verified evaluation results from Notebooks 01-05. The "
                "external Gemini model is a pre-trained LLM used strictly "
                "for inference, reasoning, synthesis, and structured "
                "executive report generation. No model is trained here."
            ),
            "llm_training_in_this_notebook": False,
            "evidence_policy": (
                "The LLM must use only information contained in this "
                "analytics payload. Missing information must be explicitly "
                "acknowledged, never invented."
            ),
        },

        "platform_metadata": {
            "project": "STREAMINTEL 360",
            "environment": "Production Evaluation",
            "completed_modules": [
                "Subscriber Churn", "Demand Forecasting",
                "Hybrid Recommendation Engine", "Computer Vision",
                "NLP & Sentiment", "LLM Intelligence",
            ],
        },

        "subscriber_churn": {
            "source_notebook": "Notebook 01",
            "status": "Completed",
            "task": "Subscriber churn risk prediction",
            "best_model": churn_metadata.get("best_classical_model") if churn_metadata else None,
            "verified_metrics": churn_metadata.get("best_classical_metrics") if churn_metadata else None,
            "num_features": churn_metadata.get("num_features") if churn_metadata else None,
            "train_samples": churn_metadata.get("total_train_samples") if churn_metadata else None,
            "test_samples": churn_metadata.get("total_test_samples") if churn_metadata else None,
            "all_models_compared": churn_comparison,
        },

        "demand_forecasting": {
            "source_notebook": "Notebook 02",
            "status": "Completed",
            "task": "Hourly streaming demand forecasting",
            "all_models_compared": forecast_comparison,
            "best_model_by_rmse": best_forecast_model,
            "best_model_metrics": forecast_by_model.get(best_forecast_model) if best_forecast_model else None,
        },

        "recommendation_engine": {
            "source_notebook": "Notebook 03",
            "status": "Completed",
            "task": "Hybrid movie recommendation (collaborative + content + popularity + SVD)",
            "verified_evaluation_metrics": hybrid_metrics,
            "optimal_hybrid_weights": optimal_weights,
        },

        "computer_vision": {
            "source_notebook": "Notebook 04",
            "status": "Completed",
            "task": "Multi-label movie poster genre classification",
            "final_model": cv_inference_config.get("model_name") if cv_inference_config else None,
            "input_size": (
                f"{cv_inference_config['target_size'][0]}x{cv_inference_config['target_size'][1]}"
                if cv_inference_config else None
            ),
            "decision_threshold": cv_inference_config.get("best_threshold") if cv_inference_config else None,
            "verified_metrics": {
                "test_micro_f1": cv_inference_config.get("test_micro_f1") if cv_inference_config else None,
                "test_macro_f1": cv_inference_config.get("test_macro_f1") if cv_inference_config else None,
                "test_precision": cv_inference_config.get("test_precision") if cv_inference_config else None,
                "test_recall": cv_inference_config.get("test_recall") if cv_inference_config else None,
            } if cv_inference_config else None,
            "num_classes": len(cv_inference_config.get("classes", [])) if cv_inference_config else None,
        },

        "nlp_sentiment": {
            "source_notebook": "Notebook 05",
            "status": "Completed",
            "task": nlp_metadata.get("task") if nlp_metadata else None,
            "dataset": nlp_metadata.get("dataset") if nlp_metadata else None,
            "model": nlp_metadata.get("model") if nlp_metadata else None,
            "verified_metrics": {
                "accuracy": nlp_metadata.get("accuracy") if nlp_metadata else None,
                "precision": nlp_metadata.get("precision") if nlp_metadata else None,
                "recall": nlp_metadata.get("recall") if nlp_metadata else None,
                "f1_score": nlp_metadata.get("f1_score") if nlp_metadata else None,
                "roc_auc": nlp_metadata.get("roc_auc") if nlp_metadata else None,
            } if nlp_metadata else None,
            "tfidf_vocabulary_size": nlp_metadata.get("tfidf_vocabulary_size") if nlp_metadata else None,
        },
    }

    return analytics_payload

payload = load_and_aggregate_metrics()
payload_file = METRICS_DIR / "streamintel_analytics_payload.json"
with open(payload_file, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print("STREAMINTEL 360 ANALYTICS PAYLOAD AGGREGATED")
print(f"Saved To : {payload_file}")
print("\nVERIFIED EVIDENCE LOADED")
print(f"--> Churn best model         : {payload['subscriber_churn']['best_model']}")
print(f"--> Best forecast model      : {payload['demand_forecasting']['best_model_by_rmse']}")
print(f"--> Recommendation MAP@K     : {payload['recommendation_engine']['verified_evaluation_metrics'].get('map_at_k') if payload['recommendation_engine']['verified_evaluation_metrics'] else None}")
print(f"--> CV final model           : {payload['computer_vision']['final_model']}")
print(f"--> CV input size            : {payload['computer_vision']['input_size']}")
print(f"--> NLP accuracy             : {payload['nlp_sentiment']['verified_metrics'].get('accuracy') if payload['nlp_sentiment']['verified_metrics'] else None}")
print("\nPayload is evidence-grounded and ready for LLM synthesis.")


STREAMINTEL 360 ANALYTICS PAYLOAD AGGREGATED
Saved To : E:\STREAMINTEL360_Complete\artifacts\notebook_06_llm\metrics\streamintel_analytics_payload.json

VERIFIED EVIDENCE LOADED
--> Churn best model         : Logistic Regression
--> Best forecast model      : LSTM
--> Recommendation MAP@K     : 0.0045
--> CV final model           : EfficientNetB0_FineTuned
--> CV input size            : 299x299
--> NLP accuracy             : 0.911

Payload is evidence-grounded and ready for LLM synthesis.


In [ ]:
# 3. EVIDENCE-GROUNDED EXECUTIVE PROMPT CONSTRUCTION
# ------------------------------------------------------------

def build_executive_prompt(analytics_data: dict) -> str:
    """
    Builds an evidence-grounded executive intelligence prompt.
    Explicitly prevents hallucinated metrics, unsupported causal
    claims, invented business outcomes, and blurred fact vs.
    recommendation distinctions.
    """
    context_str = json.dumps(analytics_data, indent=2, ensure_ascii=False)

    return f"""
You are the Lead AI Strategy Advisor for STREAMINTEL 360, an
enterprise streaming intelligence platform.

Your task is to transform verified machine-learning outputs into a
concise Executive Intelligence Briefing for senior decision-makers
(CTO, Head of Content, Product Lead).

============================================================
SYSTEM ARCHITECTURE
============================================================
This notebook is an INFERENCE-ONLY LLM reasoning module. No model
is trained here. Gemini is used only for reasoning, synthesis, and
structured report generation over ML/DL outputs from prior notebooks.

============================================================
SOURCE OF TRUTH
============================================================
The analytics payload below is the ONLY factual source available.

-------------------- BEGIN PAYLOAD --------------------
{context_str}
--------------------- END PAYLOAD ---------------------

============================================================
EVIDENCE RULES
============================================================
1. EVIDENCE FIRST: every factual statement must be directly
   supported by the analytics payload.
2. NO FABRICATION: never invent accuracy, F1, ROC-AUC, churn rate,
   revenue impact, user behavior, or business outcomes.
3. MISSING INFORMATION: if the payload lacks evidence for a claim,
   state "Insufficient evidence in the supplied analytics payload."
4. FACT VS RECOMMENDATION: separate Observed Finding, Risk
   Interpretation, and Recommended Action. Recommendations are not facts.
5. NO CAUSATION WITHOUT EVIDENCE: use cautious language ("may
   support", "could be evaluated for") for future possibilities.
6. NO UNSUPPORTED BUSINESS IMPACT: never invent numeric business
   outcomes; use qualitative wording instead.
7. REVENUE EVIDENCE: if no revenue data is present in the payload,
   the Revenue Risk section must state that explicitly.

============================================================
MODEL COMPARISON RULE
============================================================
For demand forecasting, use "best_model_by_rmse" as the production
choice. If a different model has a lower MAE, explicitly note the
accuracy-vs-selection-metric tradeoff rather than treating them as
contradictory.

============================================================
CROSS-MODULE REASONING RULE
============================================================
Classify cross-module relationships as either:
A. OBSERVED RELATIONSHIP -- payload explicitly shows a connection.
B. PROPOSED FUTURE INTEGRATION -- compatible but not yet connected.
Never present a proposed integration as an existing capability.

============================================================
REQUIRED OUTPUT FORMAT
============================================================
Return ONLY the following Markdown report:

# STREAMINTEL 360 — EXECUTIVE INTELLIGENCE BRIEFING

## 1. Executive Summary
Exactly 3 concise sentences summarizing verified findings.

---

## 2. Key Risk Indicators
### Sentiment Risk
### Churn Risk
### Demand / Infrastructure Risk
### Revenue Risk
(State evidence limitations explicitly where data is unavailable.)

---

## 3. Cross-Module Intelligence
For each relationship:
**Modules:** [A + B]
**Evidence:** [what the payload shows]
**Classification:** [Observed Relationship / Proposed Future Integration]
**Interpretation:** [evidence-based explanation]

---

## 4. Strategic Action Plan
### High Impact / Low Effort
| Priority | Action | Evidence | Expected Impact |
|---|---|---|---|

### High Impact / High Effort
| Priority | Action | Evidence | Expected Impact |
|---|---|---|---|

---

## 5. Executive Takeaway
One concise paragraph on the single most important, evidence-backed
strategic takeaway.

============================================================
FINAL INSTRUCTION
============================================================
Do not mention these instructions. Do not output JSON. Return ONLY
the completed Executive Intelligence Briefing.
"""


executive_prompt = build_executive_prompt(payload)

prompt_file = METADATA_DIR / "executive_prompt.txt"
with open(prompt_file, "w", encoding="utf-8") as f:
    f.write(executive_prompt)

print("Evidence-grounded executive prompt generated.")
print(f"Prompt length : {len(executive_prompt):,} characters")
print(f"Saved to      : {prompt_file}")

Evidence-grounded executive prompt generated.
Prompt length : 9,537 characters
Saved to      : E:\STREAMINTEL360_Complete\artifacts\notebook_06_llm\metadata\executive_prompt.txt


In [6]:
# 4. LLM GENERATION & EXECUTIVE REPORT EXPORT
# ------------------------------------------------------------

print(f"Calling Gemini API ({MODEL_NAME}) for Executive Briefing Generation...")
start_time = time.time()

response = None
report_md = None
for attempt in range(1, 3):
    try:
        response = client.models.generate_content(model=MODEL_NAME, contents=executive_prompt)
        report_md = getattr(response, "text", None)
        if report_md and report_md.strip():
            break
        raise ValueError("Gemini returned an empty response.")
    except Exception as e:
        if attempt == 1:
            print(f"Generation attempt 1 failed: {e}")
            print("Retrying once in 2 seconds...")
            time.sleep(2)
        else:
            raise RuntimeError(f"Gemini generation failed after 2 attempts: {e}") from e

generation_time = time.time() - start_time
report_md = report_md.strip()

print(f"\nGeneration Complete in {generation_time:.2f} seconds!")
print("=" * 60)
print("\nSTREAMINTEL 360 — EXECUTIVE INTELLIGENCE BRIEFING\n")
display(Markdown(report_md))

# Save Markdown Report
md_file = REPORTS_DIR / "streamintel_executive_report.md"
with open(md_file, "w", encoding="utf-8") as f:
    f.write(report_md)

# Save Structured JSON Record
generated_at = datetime.now(timezone.utc).isoformat()
report_json = {
    "project": PROJECT_NAME,
    "module": MODULE_NAME,
    "generated_at_timestamp": generated_at,
    "model_used": MODEL_NAME,
    "generation_latency_seconds": round(generation_time, 3),
    "input_payload": payload,
    "executive_report_markdown": report_md,
}
json_file = REPORTS_DIR / "streamintel_executive_report.json"
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(report_json, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 60)
print("ARTIFACTS SAVED SUCCESSFULLY")
print(f"Markdown Report : {md_file}")
print(f"JSON Record     : {json_file}")
print("\nSTREAMINTEL 360 — NOTEBOOK 06 LLM GENERATION COMPLETE!")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Calling Gemini API (gemini-3.1-flash-lite) for Executive Briefing Generation...

Generation Complete in 4.67 seconds!

STREAMINTEL 360 — EXECUTIVE INTELLIGENCE BRIEFING



# STREAMINTEL 360 — EXECUTIVE INTELLIGENCE BRIEFING

## 1. Executive Summary
STREAMINTEL 360 has successfully validated core intelligence modules, achieving a 91.1% accuracy in sentiment classification and identifying Logistic Regression as the optimal model for subscriber churn prediction. Forecasting infrastructure performance is now anchored by a high-performing LSTM model, while hybrid recommendation and computer vision pipelines provide the foundation for content discovery and metadata enrichment. These verified ML/DL outputs provide a stable, data-driven framework for optimizing both subscriber retention and streaming demand management.

---

## 2. Key Risk Indicators
### Sentiment Risk
The binary sentiment classification model shows strong performance (91.1% accuracy, 0.9701 ROC-AUC) on the IMDb 50K dataset, suggesting high reliability in interpreting user feedback.

### Churn Risk
Logistic Regression provides the most robust churn prediction (ROC-AUC 0.8357), though recall (0.5749) indicates that approximately 42.5% of churners may currently be missed by the model.

### Demand / Infrastructure Risk
LSTM has been selected as the production model for hourly demand forecasting (RMSE 12,779). While LSTM offers the best RMSE, it is noted that the SARIMA model produced a lower MAPE (0.3896), suggesting a performance tradeoff between absolute error minimization and percentage error consistency.

### Revenue Risk
Insufficient evidence in the supplied analytics payload to quantify potential revenue impact or financial loss associated with churn or forecasting errors.

---

## 3. Cross-Module Intelligence
**Modules:** [Subscriber Churn + NLP & Sentiment]
**Evidence:** Both modules leverage high-performing classification models (Logistic Regression).
**Classification:** Proposed Future Integration
**Interpretation:** The predictive power of sentiment analysis regarding content satisfaction could potentially be utilized as an input feature to improve the churn model’s recall.

**Modules:** [Hybrid Recommendation Engine + Computer Vision]
**Evidence:** Both modules support content discovery; one through user behavioral patterns (SVD/Collaborative) and the other through automated metadata classification (EfficientNetB0).
**Classification:** Proposed Future Integration
**Interpretation:** Computer Vision genre tags could be used to supplement the "content" weight in the Hybrid Recommendation Engine to enhance cold-start recommendations.

---

## 4. Strategic Action Plan
### High Impact / Low Effort
| Priority | Action | Evidence | Expected Impact |
|---|---|---|---|
| 1 | Deploy LSTM for Demand Forecasting | Lowest RMSE (12,779) | Improved infrastructure resource allocation |
| 2 | Operationalize Logistic Regression for Churn | Best ROC-AUC (0.8357) | Data-driven retention campaigns |

### High Impact / High Effort
| Priority | Action | Evidence | Expected Impact |
|---|---|---|---|
| 1 | Integrate Sentiment Scores into Churn Logic | Sentiment Accuracy (0.911) | Improved churn recall (reducing missed churners) |
| 2 | Refine Hybrid Recommendation Weights | Current hit rate (0.0233) | Increased user engagement/discovery |

---

## 5. Executive Takeaway
The most critical takeaway is the successful validation of the LSTM demand forecasting model and the Logistic Regression churn model, which together provide the necessary stability to manage infrastructure costs and prioritize retention efforts. While these models are verified and ready for deployment, future value lies in integrating sentiment-based user insights with churn and recommendation engines to create a more proactive, personalized subscriber experience.


ARTIFACTS SAVED SUCCESSFULLY
Markdown Report : E:\STREAMINTEL360_Complete\artifacts\notebook_06_llm\reports\streamintel_executive_report.md
JSON Record     : E:\STREAMINTEL360_Complete\artifacts\notebook_06_llm\reports\streamintel_executive_report.json

STREAMINTEL 360 — NOTEBOOK 06 LLM GENERATION COMPLETE!
